Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name, email and UFID.
Please do not modify instruction cells or any cells with automated tests (marked with `[ASSERTS]`). Note: you can add new cells if you need them, but answers must be in the cells with `YOUR CODE HERE` or "YOUR ANSWER HERE" comments.

---

## Homework 5: CNNs, RNNs, and AutoEncoders

## Preamble: Write your Name, Email and UFID

In [ ]:
NAME = 'Your name here.'
EMAIL = 'Your email here.'
UFID = 12345678

# YOUR CODE HERE
raise NotImplementedError()

print('Homework 5 -- name: {}, email: {}, UFID: {}\n'.format(NAME, EMAIL, UFID))

In [ ]:
""" [ASSERTS] Check that your name, email, and UFID is filled in."""
assert NAME != '' and NAME != 'Your name here.' and len(NAME) > 3
assert EMAIL != '' and EMAIL != 'Your email here.' and len(EMAIL) > 7
assert type(UFID) == int and UFID != 12345678 and UFID >= 10000000 and UFID <= 99999999

## Academic Integrity

### <span style="color:red;">This is an individual assignment. Academic integrity violations (i.e., cheating, plagiarism) will be reported to SCCR!</span><br/>
#### The official CISE policy recommended for such offenses is a course grade of E. Additional sanctions may be imposed by SCCR such as marks on your permanent educational transcripts, dismissal or expulsion.
#### Reminder of the Honor Pledge: On all work submitted for credit by Students at the University of Florida, the following pledge is either required or implied: *"On my honor, I have neither given nor received unauthorized aid in doing this assignment."*

#### Acknowledgement: Do you acknowledge and understand the academic integrity warning above? 

In [ ]:
academic_integrity_acknowledgement = False
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
""" [ASSERTS] Check that you acknowledge the academic integrity warning, you understand it and have been reminded of the UF Honor Pledge."""
assert academic_integrity_acknowledgement

#### The following cell's code (import statements etc.) is provided for you and you should not need to change it.

In [ ]:
# Load packages we need
import sys
import os
import time

import numpy as np
import pandas as pd
import sklearn

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader

from torchvision import datasets, transforms
from torchvision.datasets import FashionMNIST, CIFAR10

# Note: we will load the IMDB dataset from tensorflow.keras...
from tensorflow.keras.datasets import imdb

from sklearn.model_selection import train_test_split

from torch.nn import Linear, Conv2d, MaxPool2d, Dropout, Flatten, ReLU
from sklearn.model_selection import train_test_split

from matplotlib import pyplot as plt
plt.rcParams.update({'font.size': 14})


# Let's check our software versions
print('------------')
print('### Python version: ' + __import__('sys').version)
print('### NumPy version: ' + np.__version__)
print('### Scikit-learn version: ' + sklearn.__version__)
print('### PyTorch version: ' + torch.__version__)
print('------------')

def var_exists(var_name):
    return (var_name in globals() or var_name in locals())


#### This is the seed we will use, do not change it.

In [ ]:
# set the seed
seed = 42
np.random.seed(seed)

prop_vec = [16, 2, 2] # proportions for train - val - test splits

epsf = 1e-9 # small epsilon value for floating point comparisons

In [ ]:
""" [ASSERTS] Check seed. """
assert seed == 42

In [ ]:
"""
## Plots a set of images (all m x m)
## input is  a square number of images, i.e., np.array with shape (z*z, dim_x, dim_y) for some integer z > 1
"""
def plot_images(im, dim_x=28, dim_y=28, one_row=False, out_fp='out.png', save=False, show=True, cmap='gray', fig_size=(14,14), titles=None, titles_fontsize=12):
    fig = plt.figure(figsize=fig_size)
    im = im.reshape((-1, dim_x, dim_y))

    num = im.shape[0]
    assert num <= 3 or np.sqrt(num)**2 == num or one_row, 'Number of images is too large or not a perfect square!'
    
    if titles is not None:
        assert num == len(titles)
    
    if num <= 3:
        for i in range(0, num):
            plt.subplot(1, num, 1 + i)
            plt.axis('off')
            if type(cmap) == list:
                assert len(cmap) == num
                plt.imshow(im[i], cmap=cmap[i]) # plot raw pixel data
            else:
                plt.imshow(im[i], cmap=cmap) # plot raw pixel data
            if titles is not None:
                plt.title(titles[i], fontsize=titles_fontsize)
    else:
        sq = int(np.sqrt(num))
        for i in range(0, num):
            if one_row:
                plt.subplot(1, num, 1 + i)
            else:
                plt.subplot(sq, sq, 1 + i)
            plt.axis('off')
            if type(cmap) == list:
                assert len(cmap) == num
                plt.imshow(im[i], cmap=cmap[i]) # plot raw pixel data
            else:
                plt.imshow(im[i], cmap=cmap) # plot raw pixel data
            if titles is not None:
                plt.title(titles[i], fontsize=titles_fontsize)

    if save:
        plt.savefig(out_fp)

    if show:
        plt.show()
    else:
        plt.close()

### Loading CIFAR-10 data.

In [ ]:
def load_preprocess_cifar10(onehot=False, minmax_normalize=True):
    # Define class labels
    labels = np.array(['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'])
    num_classes = labels.shape[0]
    
    # Load CIFAR-10 training and test sets using torchvision
    # CIFAR10.data is a numpy array of shape (N, 32, 32, 3)
    train_dataset = CIFAR10(root='./data', train=True, download=True)
    testval_dataset = CIFAR10(root='./data', train=False, download=True)
    
    train_x = train_dataset.data   # shape: (50000, 32, 32, 3)
    train_y = np.array(train_dataset.targets)  # shape: (50000,)
    
    testval_x = testval_dataset.data   # shape: (10000, 32, 32, 3)
    testval_y = np.array(testval_dataset.targets)  # shape: (10000,)
    
    if minmax_normalize:
        train_x = train_x.astype(np.float32) / 255.0
        testval_x = testval_x.astype(np.float32) / 255.0
        
    if onehot:
        train_y = np.eye(num_classes)[train_y]  # becomes (50000, 10)
        testval_y = np.eye(num_classes)[testval_y]  # becomes (10000, 10)
    
    # Split testval into validation and test sets
    nval = testval_x.shape[0] // 2  # 5000
    val_x = testval_x[:nval]
    val_y = testval_y[:nval]
    test_x = testval_x[nval:]
    test_y = testval_y[nval:]
    
    return train_x, train_y, test_x, test_y, val_x, val_y, labels

In [ ]:
# --- Sanity Checks ---
# First, load data without onehot encoding and without normalization.
train_x, train_y, test_x, test_y, val_x, val_y, labels = load_preprocess_cifar10(onehot=False, minmax_normalize=False)
assert train_x.shape[0] == train_y.shape[0] and test_x.shape[0] == test_y.shape[0] and val_x.shape[0] == val_y.shape[0]
assert np.amax(train_x) >= 255 and np.amax(test_x) >= 255 and np.amax(val_x) >= 255
assert (train_y.ndim == 1) or (train_y.shape == (train_y.shape[0],1))

# Now load data with onehot encoding (labels become vectors) but still without normalization.
train_x, train_y, test_x, test_y, val_x, val_y, labels = load_preprocess_cifar10(onehot=True, minmax_normalize=False)
assert np.amax(train_x) >= 255 and np.amax(test_x) >= 255 and np.amax(val_x) >= 255
assert train_y.shape == (train_y.shape[0], 10) and train_y.shape[1] == test_y.shape[1]

# Finally, load data with normalization and onehot encoding.
train_x, train_y, test_x, test_y, val_x, val_y, labels = load_preprocess_cifar10(onehot=True)
assert np.amax(train_x) <= 1 and np.amax(test_x) <= 1 and np.amax(val_x) <= 1
assert np.amin(train_x) >= 0 and np.amin(test_x) >= 0 and np.amin(val_x) >= 0

assert labels.shape[0] == 10 and labels.shape[0] == train_y.shape[1]
print(train_x.shape, val_x.shape, test_x.shape, train_y.shape)

---
# [Task 1] (25 points) Training a CNN For CIFAR-10

#### We will use the following architecture
- Conv layer with 32 filters, (3,3) filter size, stride of 1, padding 'same'
- Conv layer with 32 filters, (3,3) filter size, stride of 1, padding 'same'
- Max pooling layer (2,2)
- Conv layer with 64 filters, (3,3) filter size, stride of 1, padding 'same'
- Conv layer with 64 filters, (3,3) filter size, stride of 1, padding 'same'
- Max pooling layer (2,2)
- Conv layer with 128 filters, (3,3) filter size, stride of 1, padding 'same'
- Conv layer with 128 filters, (3,3) filter size, stride of 1, padding 'same'
- Max pooling layer (2,2)
- Flatten
- FC with 128 units
- Dropout with rate 40%
- FC with 96 units
- Dropout with rate 25%
- (Output layer) FC with 10 units

#### For all layers (if applicable) except the output layer you should use:
- ReLU as activation function
- LeCun uniform weight initialization strategy
- L2 regularization with regularization constant set to 0.0008

#### For the output layer you should select a suitable activation function that is consistent with the task and loss function you use. Use Adam for the optimizer with learning rate 0.0002.

#### With this architecture correctly implemented you should have 500k-600k parameters and it should be easy to achieve 73%+ test accuracy.

### [Task 1a] (15 points) Implement create_compile_cnn()

In [ ]:
def create_compile_cnn(input_shape=[32, 32, 3], num_outputs=10, verbose=False):

    # Convert input shape from channels-last (H, W, C) to channels-first (C, H, W)
    if isinstance(input_shape, (list, tuple)) and input_shape[-1] == 3:
        pt_input_shape = (input_shape[-1], input_shape[0], input_shape[1])
    else:
        pt_input_shape = input_shape

    
    # L2 regularization will be applied via weight_decay in the optimizer.
    # Helper: LeCun uniform initialization. You should use this function to perform LeCun uniform init
    def lecun_uniform(tensor):
        fan_in = nn.init._calculate_correct_fan(tensor, mode='fan_in')
        bound = np.sqrt(3.0 / fan_in)
        return nn.init.uniform_(tensor, -bound, bound)

    
    class CIFAR10CNN(nn.Module):
        def __init__(self, input_shape, num_outputs):
            super(CIFAR10CNN, self).__init__()
            # YOUR CODE HERE
            raise NotImplementedError()
        
        def forward(self, x):
            # YOUR CODE HERE
            raise NotImplementedError()
            
            return x
        

    model = CIFAR10CNN(input_shape, num_outputs)
    # Create the optimizer: Adam with learning rate 0.0002 and L2 regularization via weight_decay.
    # YOUR CODE HERE
    raise NotImplementedError()
    
    if verbose:
        print(model)

    model.optimizer = opt
    model.loss_func = nn.CrossEntropyLoss()
    
    return model


In [ ]:
_ = create_compile_cnn(verbose=True)

### [Task 1b] (10 points) Train the model. Fill in the implementation below. 
#### Note: make sure to delete the model file on disk if incorrect, or else the assertions passing may not reflect a correct implementation (loading the model if it exists is there to save time and not retrain the model each time if the implementation does not change).

In [ ]:
cnn_model_fp = './cifar10-cnn.pt'
train_x, train_y, test_x, test_y, val_x, val_y, labels = load_preprocess_cifar10()
train_x = np.transpose(train_x, (0, 3, 1, 2))
val_x = np.transpose(val_x, (0, 3, 1, 2))
test_x = np.transpose(test_x, (0, 3, 1, 2))


# If the model file exists, load it. Otherwise, train and save the model.
if os.path.exists(cnn_model_fp):
    model = create_compile_cnn(verbose=False)
    model.load_state_dict(torch.load(cnn_model_fp))
else:
    model = create_compile_cnn(verbose=False)
    
    # Do the training for at least 5 epochs (here using 15 epochs) with your chosen batch size.
    # (Ensure your network is actually learning.)
    # You can set any callbacks you want (e.g., early stopping).

    max_epochs = 15
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)
    model.to(device)
    
    # Create DataLoaders from train and validation data
    train_dataset = TensorDataset(torch.tensor(train_x, dtype=torch.float32), torch.tensor(train_y, dtype=torch.long))
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    val_dataset = TensorDataset(torch.tensor(val_x, dtype=torch.float32), torch.tensor(val_y, dtype=torch.long))
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

    """Fill in your code here. You will need to write a simple training loop in pytorch.
    """
    # YOUR CODE HERE
    raise NotImplementedError()
    
    # Save the model state dictionary.
    torch.save(model.state_dict(), cnn_model_fp)

In [ ]:
""" [ASSERTS] Check 1a and 1b completed. """

# Evaluate the model on the test data (manual evaluation)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Convert test data to torch tensors
# Note: test_x should be in channel-first format, and test_y should be class indices.
test_tensor = torch.tensor(test_x, dtype=torch.float32).to(device)

# If test_y is one-hot, convert to class indices:
if test_y.ndim > 1:
    test_labels = torch.tensor(np.argmax(test_y, axis=1), dtype=torch.long).to(device)
else:
    test_labels = torch.tensor(test_y, dtype=torch.long).to(device)

with torch.no_grad():
    outputs = model(test_tensor)
    loss = model.loss_func(outputs, test_labels).item()
    preds = torch.argmax(outputs, dim=1)
    acc = (preds == test_labels).float().mean().item()

print('[Model] Test accuracy: {:.2f}%'.format(100*acc)) # this should print ~73%

assert var_exists('model')

# Recreate the model (in case the version on disk is no good)
model = create_compile_cnn(verbose=False)

# Count the trainable parameters
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of training params: {trainable_count}.")
assert trainable_count > 500000 and trainable_count < 600000

layers = list(model.children())
assert len(layers) == 16, f"Expected 16 layers, got {len(layers)}"

first_conv = None
for m in model.modules():
    if isinstance(m, nn.Conv2d):
        first_conv = m
        break
        
assert first_conv is not None, "No Conv2d layer found."
assert first_conv.out_channels == 32, f"Expected 32 filters, got {first_conv.out_channels}"
assert first_conv.groups == 1, f"Expected groups==1, got {first_conv.groups}"


---
# [Task 2] (25 points) Processing Sequence Data with RNNs for Sentiment Analysis

## We will use the imdb dataset for this task. The dataset consists of reviews from IMDB. We will use it for sentiment analysis.

## [Task 2a] (10 points) Fill in the implementation of load_preprocess_imdb()

In [ ]:
# this is loaded from Keras
def load_preprocess_imdb(num_words=12000, train_prop=0.8, val_prop=0.5, maxlen=100, vectorize=False):

    # Load the data using the Keras IMDB loader.
    (train_x, train_y), (testval_x, testval_y) = imdb.load_data(num_words=num_words, oov_char=0)

    """ Implement a function to pad each sequence within x to maxlen (pad with 0s) and return an np.array. 
        If a sequence is longer than maxlen, truncate it. """
    def seq_to_padded_array(x):
        # YOUR CODE HERE
        raise NotImplementedError()

    # Pad training and testval sequences.
    train_x = seq_to_padded_array(train_x)
    testval_x = seq_to_padded_array(testval_x)
    
    # Merge the training and testval splits.
    all_x = np.concatenate([train_x, testval_x], axis=0)
    all_y = np.concatenate([train_y, testval_y], axis=0)
    
    # Split the merged data into train and testval, then split testval into validation and test.
    train_x, testval_x, train_y, testval_y = train_test_split(all_x, all_y, test_size=1-train_prop, random_state=seed)
    val_x, test_x, val_y, test_y = train_test_split(testval_x, testval_y, test_size=1-val_prop, random_state=seed)
    
    return train_x, train_y, test_x, test_y, val_x, val_y

In [ ]:
vocab_size = 12000 # size of the vocabulary
maxlen = 150 

# sanity checks
train_x, train_y, test_x, test_y, val_x, val_y = load_preprocess_imdb(num_words=vocab_size, maxlen=maxlen)
print(train_x.shape, train_y.shape, test_x.shape, test_y.shape, val_x.shape, val_y.shape)

In [ ]:
""" [ASSERTS] Check 2a completed. """
assert var_exists('train_x') and var_exists('train_y')
assert train_x.shape == (40000, maxlen) and train_y.shape == (train_x.shape[0],)


### [Task 2b] (15 points) Complete the code below to define an RNN architecture for sentiment analysis. The goal is to predict the sentiment of IMDB reviews.

### Use the following architecture. For whatever is not specified (loss function, clamping, etc.) you should complete it as appropriate.
- Embedding layer of size 'embedding_size'
- GRU layer with 64 units
- GRU layer with 64 units and dropout rate 20%
- Dense layer with num_outputs units

### With this architecture correctly implemented you should have about 2m parameters and it should be easy to achieve 84%+ test accuracy.

In [ ]:
def create_compile_rnn(input_shape=[None], vocab_size=12000, embedding_size=196, num_outputs=1, verbose=False):
    
    class RNNModel(nn.Module):
        def __init__(self, vocab_size, embedding_size, num_outputs):
            super(RNNModel, self).__init__()
            # YOUR CODE HERE
            raise NotImplementedError()
            
        def forward(self, x):
            # x: (batch, seq_len) of word indices.
            # YOUR CODE HERE
            raise NotImplementedError()
            return x

    model = RNNModel(vocab_size, embedding_size, num_outputs)
    opt = optim.Adam(model.parameters(), lr=0.0008)
    

    if verbose:
        print(model)
        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print("Total parameters:", total_params)
    
    # Attach optimizer and loss function to the model.
    # Since our model's output is passed through sigmoid, we use BCE loss.
    model.optimizer = opt
    model.loss_func = nn.BCELoss()
    
    return model

In [ ]:
model = create_compile_rnn(verbose=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model.to(device)

# Create DataLoaders for train and validation sets.
train_dataset = TensorDataset(torch.tensor(train_x, dtype=torch.float32),
    torch.tensor(train_y, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

val_dataset = TensorDataset(torch.tensor(val_x, dtype=torch.float32),
    torch.tensor(val_y, dtype=torch.long))
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

# Set number of epochs and early stopping parameters.
# YOUR CODE HERE
raise NotImplementedError()

best_val_loss = float('inf')
patience_counter = 0
hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(max_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for inputs, labels in train_loader:
         inputs = inputs.to(device)
         # Convert labels to float and reshape to (batch, 1) for BCELoss
         labels = labels.to(device).float()
         model.optimizer.zero_grad()
         outputs = model(inputs)  # outputs shape: (batch, 1)
         loss = model.loss_func(outputs, labels)
         loss.backward()
         model.optimizer.step()
         running_loss += loss.item() * inputs.size(0)
         
         # Compute training accuracy: threshold outputs at 0.5
         preds = (outputs.view(-1) > 0.5).float()
         true_labels = labels.view(-1)
         correct_train += (preds == true_labels).sum().item()
         total_train += true_labels.size(0)
        
    train_loss = running_loss / len(train_loader.dataset)
    train_acc = correct_train / total_train
    
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
         for inputs, labels in val_loader:
             inputs = inputs.to(device)
             labels = labels.to(device).float()
             outputs = model(inputs)
             loss = model.loss_func(outputs, labels)
             running_val_loss += loss.item() * inputs.size(0)
             
             preds = (outputs.view(-1) > 0.5).float()
             true_labels = labels.view(-1)
             correct_val += (preds == true_labels).sum().item()
             total_val += true_labels.size(0)
    val_loss = running_val_loss / len(val_loader.dataset)
    val_acc = correct_val / total_val
    
    hist["train_loss"].append(train_loss)
    hist["val_loss"].append(val_loss)
    hist["train_acc"].append(train_acc)
    hist["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Train Acc {100*train_acc:.2f}%, Val Loss {val_loss:.4f}, Val Acc {100*val_acc:.2f}%")

In [ ]:
""" [ASSERTS] Check 2b completed. """
# Evaluate the model on the test data:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Convert test_x and test_y to torch tensors.
import torch
test_tensor = torch.tensor(test_x, dtype=torch.float32).to(device)
# Assume test_y are class indices; if one-hot, convert with np.argmax.
if test_y.ndim > 1:
    test_labels = torch.tensor(np.argmax(test_y, axis=1), dtype=torch.long).to(device)
else:
    test_labels = torch.tensor(test_y, dtype=torch.long).to(device)
test_labels = test_labels.float()
with torch.no_grad():
    outputs = model(test_tensor)
    preds = (outputs > 0.5).long()
    correct = (preds == test_labels).sum().item()
    total = test_labels.size(0)
    acc = correct / total
    loss = model.loss_func(outputs, test_labels).item()

print('[Model] Test accuracy: {:.2f}%'.format(100 * acc))


---
# [Task 3] (25 points) Sentiment Analysis with Multi-Head Attention

## In this task the objective is to use multi-head attention (layers.MultiHeadAttention) to match or exceed the performance of the RNN trained in Task 2.
## You can use whatever architecture you want as long as it includes at least one MultiHeadAttention layer (with at least two heads). Ideally we also don't want it to be an RNN.
## The catch is we want to do it using much fewer total parameters.
## For example, we will aim to use less than 500k trainable parameters while still achieving 84%+ test/val accuracy.
## You will also need to ensure the total training time is less than 10 minutes.

## Hints: 
- a possible approach is to use an embedding layer appropriately connected to the MultiHeadAttention, but note that if the embedding layer cannot be too large or it will use up too many parameters. 
- you may also find it useful to use dropout.

In [ ]:
def create_compile_attention(input_shape=(150,), vocab_size=vocab_size, num_outputs=1, verbose=False):
    
    # YOUR CODE HERE
    raise NotImplementedError()

    class ImdbAttention(nn.Module):
        def __init__(self, vocab_size, embedding_size, num_outputs, seq_length):
            # YOUR CODE HERE
            raise NotImplementedError()
        
        def forward(self, x):
            # YOUR CODE HERE
            raise NotImplementedError()
            return x

    seq_length = input_shape[0]
    model = ImdbAttention(vocab_size, embedding_size, num_outputs, seq_length)
    # YOUR CODE HERE
    raise NotImplementedError()

    if verbose:
        print(model)
    
    model.optimizer = opt
    model.loss_func = nn.BCEWithLogitsLoss()
    
    return model

In [ ]:
model = create_compile_attention(verbose=True)

# Check number of parameters.
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable params: {trainable_count}.")
assert trainable_count <= 500000

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
print("Validation accuracy history:", hist["val_acc"])
print("Training accuracy history:", hist["train_acc"])
assert var_exists('model') and var_exists('hist')

# Get final epoch accuracies from the history.
final_train_acc = hist["train_acc"][-1]  # Last epoch training accuracy.
final_val_acc = hist["val_acc"][-1]      # Last epoch validation accuracy.

# Check that final validation accuracy is at least 82% and final training accuracy is above 80%.
assert final_val_acc >= 0.82, f"Final validation accuracy {final_val_acc:.4f} is below 82%."
assert final_train_acc >= 0.80, f"Final training accuracy {final_train_acc:.4f} is below 80%."

# Check number of parameters.
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable_count <= 500000

# Check total training time.
assert elapsed_time <= 600

# Evaluate the model on the test data.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Convert test data to torch tensors.
test_tensor = torch.tensor(test_x, dtype=torch.long).to(device)
# If test_y is one-hot, convert it to class indices; otherwise assume it's already in index form.
if test_y.ndim > 1:
    test_labels = torch.tensor(np.argmax(test_y, axis=1), dtype=torch.long).to(device)
else:
    test_labels = torch.tensor(test_y, dtype=torch.long).to(device)

with torch.no_grad():
    outputs = model(test_tensor)
    
    preds = (outputs.view(-1) > 0.5).long()
    correct = (preds == test_labels).sum().item()
    total = test_labels.size(0)
    acc = correct / total
    # Compute loss (for BCEWithLogitsLoss, both inputs should be floats).
    loss = model.loss_func(outputs.view(-1), test_labels.float()).item()

acc_per_100kparam_score = acc / (trainable_count / (100*1000))
print('[Model] Test accuracy: {:.2f}% [score: {:.2f}]'.format(100 * acc, 100 * acc_per_100kparam_score))


### How high of a score did you achieve? (It is possible to match the RNN performance and achieve scores above 50.)
#### Note: there is no minimum score you must achieve. The point of this task is to show you that with (multi-head) attention we can have much fewer parameters while matching or outperforming RNNs (and also faster training).

---
# [Task 4] (25 points) AutoEncoders

### We will use Fashion-MNIST for this.

In [ ]:
def load_preprocess_fmnist_data(flatten=True, onehot=True, minmax_normalize=True, val_prop=0.5, seed=None, verbose=False):
    
    # Download Fashion-MNIST using torchvision
    train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
    test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
    
    # Convert the datasets' tensors to NumPy arrays.
    # Note: train_dataset.data is of shape (60000, 28, 28) and is a uint8 tensor.
    x_train = train_dataset.data.numpy()    # shape: (60000, 28, 28)
    y_train = train_dataset.targets.numpy()   # shape: (60000,)
    x_test = test_dataset.data.numpy()        # shape: (10000, 28, 28)
    y_test = test_dataset.targets.numpy()       # shape: (10000,)
    
    # Normalize pixel values to [0,1] if requested.
    if minmax_normalize:
        x_train = x_train.astype(np.float32) / 255.0
        x_test = x_test.astype(np.float32) / 255.0
    
    if verbose:
        print('Loaded Fashion-MNIST data; shape: {} [y: {}], test shape: {} [y: {}]'.format(
            x_train.shape, y_train.shape, x_test.shape, y_test.shape))
    
    # Flatten images if requested.
    if flatten:
        flat_vec_size = 28 * 28
        x_train = x_train.reshape(x_train.shape[0], flat_vec_size)
        x_test = x_test.reshape(x_test.shape[0], flat_vec_size)
    
    # Convert labels to one-hot encoding if requested.
    if onehot:
        num_classes = 10
        y_train = np.eye(num_classes)[y_train]
        y_test = np.eye(num_classes)[y_test]
    
    train_x = x_train
    train_y = y_train
    
    # Split the test set into validation and test sets.
    testval_x = x_test
    testval_y = y_test  
    val_x, test_x, val_y, test_y = train_test_split(testval_x, testval_y, test_size=val_prop, random_state=seed)
    
    return train_x, train_y, test_x, test_y, val_x, val_y

# Grab the data (here flatten=False, onehot=False)
train_x, train_y, test_x, test_y, val_x, val_y = load_preprocess_fmnist_data(flatten=False, onehot=False, val_prop=0.5, seed=seed) 

# Sanity check shapes.
print(train_x.shape, train_y.shape, test_x.shape, test_y.shape, val_x.shape, val_y.shape)
assert train_x.shape == (60000, 28, 28) and train_y.shape == (60000,) and test_x.shape == (5000, 28, 28) and test_y.shape == (5000,)

### [Task 4a] (15 points) Complete the implementation of create_simple_ae() according to the architecture and instructions below.

#### Start by creating the encoder ('enc_model'). It should have the following architecture:
- Flatten
- FC with hidden_widths[0] units with activation ReLU
- FC with hidden_widths[1] units with activation ReLU
- Dropout with rate 'dropout_rate'
- FC with latent_width units with activation *sigmoid*

#### Then create the decoder ('dec_model'). It should have the following architecture:
- (Input has shape latent_width)
- FC with hidden_widths[1] units with activation ReLU
- FC with hidden_widths[0] units with activation ReLU
- Dropout with rate 'dropout_rate'
- FC with 784 units with activation *sigmoid*
- Reshape to input_shape

#### Finally connect the two models together into a new model ('ae_model'). Make sure that the model takes the input that the encoder takes and produces the output that the decoder produces.

In [ ]:
def create_compile_ae(input_shape=(28,28), latent_width=64, hidden_widths=[256,96], dropout_rate=0.0, verbose=False):
    import torch
    import torch.nn as nn
    import torch.optim as optim

    name = 'AE-simple'
    
    class Encoder(nn.Module):
        # YOUR CODE HERE
        raise NotImplementedError()

    class Decoder(nn.Module):
        # YOUR CODE HERE
        raise NotImplementedError()

    class AutoEncoder(nn.Module):
        def __init__(self, input_shape, hidden_widths, latent_width, dropout_rate):
            super(AutoEncoder, self).__init__()
            self.encoder = Encoder(input_shape, hidden_widths, latent_width, dropout_rate)
            self.decoder = Decoder(input_shape, hidden_widths, latent_width, dropout_rate)
            
        def forward(self, x):
            # YOUR CODE HERE
            raise NotImplementedError()

    ae_model = AutoEncoder(input_shape, hidden_widths, latent_width, dropout_rate)
    enc_model = ae_model.encoder
    dec_model = ae_model.decoder
    
    opt = optim.Adam(ae_model.parameters(), lr=0.003)
    
    if verbose:
        print(ae_model)
    
    ae_model.optimizer = opt
    ae_model.loss_func = nn.BCELoss()
    
    return name, ae_model, enc_model, dec_model

### Let's train the model -- you should get a loss < 0.28 and MSE < 0.012.

In [ ]:
# Let's train the model (no need to change any of this).
latent_width = 64
name, ae_model, enc_model, dec_model = create_compile_ae(latent_width=latent_width, verbose=True)

# Set parameters.
max_epochs = 40
batch_size = 96

train_dataset = TensorDataset(torch.tensor(train_x, dtype=torch.float32),
                                torch.tensor(train_x, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(torch.tensor(val_x, dtype=torch.float32),
                              torch.tensor(val_x, dtype=torch.float32))
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
ae_model.to(device)

# Prepare a history dictionary.
hist = {"train_loss": [], "val_loss": [], "train_mse": [], "val_mse": []}

st = time.time()
for epoch in range(max_epochs):
    ae_model.train()
    running_loss = 0.0
    running_mse = 0.0
    total_train = 0
    for inputs, targets in train_loader:
         inputs, targets = inputs.to(device), targets.to(device)
         ae_model.optimizer.zero_grad()
         outputs = ae_model(inputs)
         outputs = outputs.view(-1)  # shape: (batch,)
         loss = ae_model.loss_func(outputs, targets.view(-1))
         loss.backward()
         ae_model.optimizer.step()
         batch_size_actual = inputs.size(0)
         running_loss += loss.item() * batch_size_actual
         # Compute batch MSE
         mse = torch.mean((outputs - targets.view(-1))**2).item()
         running_mse += mse * batch_size_actual
         total_train += batch_size_actual
    train_loss = running_loss / total_train
    train_mse = running_mse / total_train
    
    ae_model.eval()
    running_val_loss = 0.0
    running_val_mse = 0.0
    total_val = 0
    with torch.no_grad():
         for inputs, targets in val_loader:
             inputs, targets = inputs.to(device), targets.to(device)
             outputs = ae_model(inputs)
             outputs = outputs.view(-1)
             loss = ae_model.loss_func(outputs, targets.view(-1))
             batch_size_actual = inputs.size(0)
             running_val_loss += loss.item() * batch_size_actual
             mse = torch.mean((outputs - targets.view(-1))**2).item()
             running_val_mse += mse * batch_size_actual
             total_val += batch_size_actual
    val_loss = running_val_loss / total_val
    val_mse = running_val_mse / total_val
    
    hist["train_loss"].append(train_loss)
    hist["val_loss"].append(val_loss)
    hist["train_mse"].append(train_mse)
    hist["val_mse"].append(val_mse)
    
    print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Train MSE {train_mse:.4f}, Val Loss {val_loss:.4f}, Val MSE {val_mse:.4f}")
    
    if val_loss < best_val_loss:
         best_val_loss = val_loss
         best_model_wts = ae_model.state_dict()
         patience_counter = 0
    else:
         patience_counter += 1
         if patience_counter >= patience:
              print("Early stopping triggered")
              break

ae_model.load_state_dict(best_model_wts)
et = time.time()
elapsed_time = et - st
print('Total training time {:.1f} seconds'.format(elapsed_time))

In [ ]:
""" [ASSERTS] Check 4a completed. """

assert var_exists('hist')
train_loss = hist["train_loss"]
val_loss = hist['val_loss']
val_mse = hist['val_mse']

assert train_loss[-1] < val_loss[-1]
assert val_loss[-1] < 0.295


### [Task 4b] (10 points) We will use plot_images() to plot the first 81 data points in the val_x. Then reconstruct val_x through the auto-encoder and plot the first 81 data points of the reconstructed data. Fill in the code in the following cell.

In [ ]:
"""Fill in your code here (~1 line)
"""
# YOUR CODE HERE
raise NotImplementedError()
plot_images(val_x_sel, dim_x=28, dim_y=28, fig_size=(10,10))

# Run the data points in 'val_x_sel' through the auto-encoder to obtain 'rec_val_x_sel'
"""Fill in your code here (~1 line)
"""
# YOUR CODE HERE
raise NotImplementedError()

# Plot the reconstructed data
plot_images(rec_val_x_sel, dim_x=28, dim_y=28, fig_size=(10,10))

### Reconstructions should look pretty good but also a bit blurry.

In [ ]:
""" [ASSERTS] Check 4b completed. """

assert var_exists('val_x_sel') and var_exists('rec_val_x_sel')
